In [1]:
import asyncio
from collections import defaultdict
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
import sys
from pathlib import Path

# Asumiendo que notebooks/ está dentro de la raíz del proyecto
root_path = Path().resolve().parent  # sube un nivel
sys.path.append(str(root_path))

In [3]:
from agents.classifier_agent import ClassifierAgent
from agents.aggregator_agent import AggregatorAgent
from data.dataset_registry import DatasetRegistry
from data.loaders.sklearn_loader import SklearnLoader
import pandas as pd

In [ ]:
"""
experiments.py — TFG: Modelado de explicaciones dinámicas en MAS

Cambios respecto a la versión original
---------------------------------------
- CORREGIDO: eliminadas referencias a mentor_log/cooldown_log (no existen)
- LIME num_samples reducido: wine=3000, digits=2000, covertype=2000
- MAX_ITER reducido a 20
- matplotlib.use("Agg") → sin ventana gráfica, más rápido
- Figuras guardadas en disco (figures/ y appendix/) en lugar de plt.show()
- plot_run_appendix usa únicamente columnas del DataFrame
"""

import asyncio, warnings, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn as nn

from sklearn.datasets      import load_wine, load_digits, fetch_covtype
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble      import RandomForestClassifier as _RFC
from sklearn.utils         import resample

from data.dataset_registry   import DatasetRegistry
from agents.classifier_agent import ClassifierAgent
from agents.aggregator_agent import AggregatorAgent
from explainers.shap_explainer import ShapExplainer
from explainers.lime_explainer import LimeExplainer
from models.random_forest       import RandomForest
from models.gradient_boosting   import GradientBoostingModel
from models.xgboost             import XGBoostModel
from models.linearSVM           import LinearSVM
from models.logistic_regression import LogisticRegressionModel
from models.kNN                 import KNNModel
from models.torch_model         import TorchModel
from visualization.visualization import build_results_dataframe

warnings.filterwarnings("ignore")
os.makedirs("results",  exist_ok=True)
os.makedirs("figures",  exist_ok=True)
os.makedirs("appendix", exist_ok=True)

# ── Constantes ────────────────────────────────────────────────────────────────

GROUPS = {
    "G1": ["rf",    "gb",   "torch"],
    "G2": ["xgb",   "svm",  "lr"],
    "G3": ["knn",   "rf",   "xgb"],
    "G4": ["torch", "svm",  "gb"],
    "G5": ["lr",    "knn",  "torch"],
}
MODEL_LABELS = {
    "rf": "RandomForest", "gb": "GradientBoosting", "xgb": "XGBoost",
    "svm": "LinearSVM",   "lr": "LogisticRegression",
    "knn": "KNN",         "torch": "TorchMLP",
}
MODEL_TYPE = {
    "rf": "Ensemble",  "gb": "Ensemble",  "xgb": "Ensemble",
    "svm": "Lineal",   "lr": "Lineal",
    "knn": "Distancia","torch": "Red neuronal",
}
LIME_SAMPLES       = {"wine": 3000, "digits": 2000, "covertype": 2000}
ABLATION_GROUP     = "G1"
CONV_ACC_THRESHOLD = 0.85
CONV_EXP_THRESHOLD = 0.60
MAX_ITER           = 20

# ── Carga de datasets ─────────────────────────────────────────────────────────

def load_datasets():
    datasets = {}
    ds = load_wine()
    datasets["wine"] = {"X_raw": ds.data, "y": ds.target,
                        "feature_names": list(ds.feature_names), "label": "Wine"}
    ds = load_digits()
    datasets["digits"] = {"X_raw": ds.data, "y": ds.target,
                          "feature_names": [f"pixel_{i}" for i in range(ds.data.shape[1])],
                          "label": "Digits"}
    ds = fetch_covtype()
    X_cv, y_cv = resample(ds.data, ds.target - 1, n_samples=5000,
                           random_state=42, stratify=ds.target - 1)
    datasets["covertype"] = {"X_raw": X_cv, "y": y_cv,
                              "feature_names": [f"f{i}" for i in range(X_cv.shape[1])],
                              "label": "Covertype"}
    return datasets

# ── Helpers ───────────────────────────────────────────────────────────────────

class ScaledLoader:
    def __init__(self, X, y, fn): self.X, self.y, self.fn = X, y, fn
    def load(self): return self.X, self.y, {"feature_names": self.fn,
                                             "n_samples": self.X.shape[0],
                                             "n_features": self.X.shape[1]}

def select_instances(X, y):
    rf = _RFC(n_estimators=100, random_state=42).fit(X, y)
    pr = rf.predict_proba(X)
    ent = -np.sum(pr * np.log(pr + 1e-8), axis=1)
    return int(np.argmax(ent)), int(np.argmax(pr.max(axis=1)))

class MLP(nn.Module):
    def __init__(self, d, nc):
        super().__init__()
        h = max(32, d * 2)
        self.net = nn.Sequential(
            nn.Linear(d, h), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(h, h//2), nn.ReLU(), nn.Linear(h//2, nc))
    def forward(self, x): return self.net(x)

def make_model(name, dim, nc):
    if name == "rf":    return RandomForest(n_estimators=10, max_depth=3, random_state=0)
    if name == "gb":    return GradientBoostingModel(n_estimators=10, learning_rate=0.3, max_depth=2)
    if name == "xgb":   return XGBoostModel(n_estimators=10, learning_rate=0.3, max_depth=2)
    if name == "svm":   return LinearSVM(C=0.01, max_iter=500)
    if name == "lr":    return LogisticRegressionModel(C=0.01, max_iter=100)
    if name == "knn":   return KNNModel(n_neighbors=10)
    if name == "torch": return TorchModel(nn_model=MLP(dim, nc),
                                          lr=1e-3, min_lr=1e-5, max_lr=0.05,
                                          epochs=15, batch_size=32)
    raise ValueError(name)

def make_explainers(feature_names, class_names, dataset_name):
    lime = LimeExplainer(feature_names=feature_names,
                         class_names=class_names,
                         discretize_continuous=False)
    n = LIME_SAMPLES.get(dataset_name, 3000)
    lime._NUM_SAMPLES = {"default": n, "torch": n}
    return [ShapExplainer(feature_names=feature_names), lime]

# ── Extracción de métricas ────────────────────────────────────────────────────

def extract_weighted_diff(history):
    return sum(
        1 for e in history
        if e["evaluation"]["majority_prediction"] !=
           max(set(r["prediction"] for r in e["responses"]),
               key=lambda p: sum(1 for r in e["responses"] if r["prediction"] == p))
    )

def extract_mentor_activations(history):
    count = dissent = 0
    for e in history:
        ev = e["evaluation"]; maj = ev["majority_prediction"]
        resp = e["responses"]; detail = ev["components"]["exp_detail"]
        q = detail.get("quality", [0.5]*len(resp))
        f = detail.get("fidelity",[0.5]*len(resp))
        c = ev["components"].get("conf", [0.0]*len(resp))
        p = [r["prediction"] for r in resp]
        aligned = [i for i,(qi,fi,ci,pi) in enumerate(zip(q,f,c,p))
                   if pi==maj and qi>=0.60 and fi>=0.60 and ci>=0.50]
        if aligned: count += 1
        else:
            dis = [i for i,(qi,fi,ci,pi) in enumerate(zip(q,f,c,p))
                   if pi!=maj and qi>=0.65 and fi>=0.70 and ci>=0.55]
            if dis: count += 1; dissent += 1
    return count, dissent

def extract_cooldown_activations(history, classifier_ids):
    total = 0
    isa = {cid: [] for cid in classifier_ids}
    dec = {cid: [] for cid in classifier_ids}
    for e in history:
        for idx, cid in enumerate(classifier_ids):
            isa[cid].append(e["responses"][idx].get("iters_since_adjust"))
            dec[cid].append(e["decisions"][idx] if idx < len(e["decisions"]) else "unknown")
    for cid in classifier_ids:
        for i in range(1, len(isa[cid])):
            p, c, d = isa[cid][i-1], isa[cid][i], dec[cid][i]
            if p is not None and c is not None and d != "keep" and c > 0 and c >= p:
                total += 1
    return total

def compute_convergence_iter(df, classifier_ids):
    result = {}
    for aid in classifier_ids:
        sub = df[df["agent"] == aid].sort_values("iteration")
        ia = sub.loc[sub["accuracy"]    >= CONV_ACC_THRESHOLD, "iteration"]
        ie = sub.loc[sub["exp_quality"] >= CONV_EXP_THRESHOLD, "iteration"]
        i_a = int(ia.iloc[0]) if not ia.empty else -1
        i_e = int(ie.iloc[0]) if not ie.empty else -1
        if i_a == -1 and i_e == -1: order = "never"
        elif i_a == -1:             order = "exp_first"
        elif i_e == -1:             order = "acc_first"
        elif abs(i_a - i_e) <= 1:  order = "simultaneous"
        elif i_a < i_e:            order = "acc_first"
        else:                      order = "exp_first"
        result[aid] = {"iter_acc": i_a, "iter_exp": i_e, "order": order}
    return result

# ── Run único ─────────────────────────────────────────────────────────────────

async def run_single(group_name, model_names, dataset_name,
                     X_scaled, y, feature_names, instance,
                     instance_label, max_iterations=MAX_ITER,
                     mentor_enabled=True, cooldown_enabled=True,
                     weighted_vote=True):

    class_names = [str(c) for c in np.unique(y)]
    num_classes = len(class_names)
    input_dim   = X_scaled.shape[1]

    registry = DatasetRegistry()
    registry.register("ds", ScaledLoader(X_scaled, y, feature_names))

    classifiers, classifier_ids = {}, []
    for mn in model_names:
        aid = f"{mn}_{group_name}_{dataset_name}"
        classifiers[aid] = ClassifierAgent(
            agent_id=aid,
            model=make_model(mn, input_dim, num_classes),
            explainers=make_explainers(feature_names, class_names, dataset_name),
            dataset_id="ds", registry=registry)
        classifier_ids.append(aid)

    aggregator = AggregatorAgent(classifier_ids=classifier_ids,
                                  max_iterations=max_iterations,
                                  background_data=X_scaled)

    if not weighted_vote:
        aggregator.evaluator._weighted_majority_vote = \
            lambda preds, accs, confs, fids=None: max(set(preds), key=preds.count)
    if not mentor_enabled:
        aggregator.feedback_builder.find_mentor = \
            lambda responses, evaluation: (None, [], False)
    if not cooldown_enabled:
        for ag in classifiers.values():
            type(ag)._cooldown_threshold = property(lambda self: 9999)

    queues = {aid: ag.inbox for aid, ag in classifiers.items()}
    queues["aggregator"] = aggregator.inbox

    await asyncio.gather(*(ag.setup() for ag in classifiers.values()))
    tasks = ([asyncio.create_task(ag.run(queues)) for ag in classifiers.values()]
             + [asyncio.create_task(aggregator.run(queues, instance))])
    await asyncio.gather(*tasks, return_exceptions=True)

    history = aggregator.global_history
    df = build_results_dataframe(aggregator_history=history,
                                  classifier_ids=classifier_ids,
                                  feature_names=feature_names)

    stop_entries = [e for e in history if e["stop"]]
    stopped_at   = stop_entries[0]["iteration"] if stop_entries else -1
    n_mentor, n_dissent = extract_mentor_activations(history)
    n_cooldown          = extract_cooldown_activations(history, classifier_ids)
    weighted_diff       = extract_weighted_diff(history)
    conv                = compute_convergence_iter(df, classifier_ids)

    mentor_iters = {e["iteration"] for e in history
                    if any((r["prediction"] == e["evaluation"]["majority_prediction"]
                             and e["evaluation"]["components"]["exp_detail"]
                                  .get("quality",[0.5]*len(history[0]["responses"]))[i] >= 0.60)
                           for i, r in enumerate(e["responses"]))}

    return {"df": df, "history": history,
            "classifier_ids": classifier_ids,
            "model_names": model_names,
            "group_name": group_name,
            "dataset_name": dataset_name,
            "instance_label": instance_label,
            "stopped_at": stopped_at,
            "n_mentor": n_mentor, "n_dissent": n_dissent,
            "n_cooldown": n_cooldown, "weighted_diff": weighted_diff,
            "convergence": conv,
            "mentor_iters": mentor_iters,
            "cooldown_iters": set()}

# ── Figuras ───────────────────────────────────────────────────────────────────

def _save(fig, name):
    for ext in ("pdf", "png"):
        path = f"figures/{name}.{ext}"
        fig.savefig(path, bbox_inches="tight", dpi=150 if ext=="png" else None)
    print(f"    → figures/{name}.pdf/.png")
    plt.close(fig)

def fig1_explainability_evolution(all_hard, dataset_names):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
    fig.suptitle("P1 — Evolución de la calidad explicativa por iteración",
                 fontsize=13, fontweight="bold")
    for ax, dn in zip(axes, dataset_names):
        frames = [r["df"] for (gn,d),r in all_hard.items() if d==dn and not r["df"].empty]
        if not frames: ax.set_title(dn); continue
        combined = pd.concat(frames)
        g = combined.groupby("iteration").agg(
            eq_m=("exp_quality","mean"), eq_s=("exp_quality","std"),
            ef_m=("exp_fidelity","mean"), ef_s=("exp_fidelity","std"),
            ec_m=("exp_consensus","mean"), ec_s=("exp_consensus","std"),
        ).reset_index()
        for m_col, s_col, label, color in [
            ("eq_m","eq_s","Exp Quality","steelblue"),
            ("ef_m","ef_s","Fidelity","coral"),
            ("ec_m","ec_s","Consensus","seagreen"),
        ]:
            ax.plot(g["iteration"], g[m_col], label=label, color=color,
                    marker="o", markersize=3, linewidth=1.5)
            ax.fill_between(g["iteration"], g[m_col]-g[s_col].fillna(0),
                            g[m_col]+g[s_col].fillna(0), color=color, alpha=0.12)
        ax.axhline(CONV_EXP_THRESHOLD, color="gray", linestyle=":", alpha=0.6)
        ax.set_title(dn, fontweight="bold"); ax.set_xlabel("Iteración")
        ax.set_ylim(0, 1.05); ax.grid(True, alpha=0.3)
        if ax is axes[0]: ax.set_ylabel("Score"); ax.legend(fontsize=8)
    plt.tight_layout(); _save(fig, "fig1_p1_evolution")

def fig2_mentor_ablation(ablation_results, dataset_names):
    conditions = ["completo","sin_mentor"]
    colors = {"completo":"steelblue","sin_mentor":"salmon"}
    labels = {"completo":"Sistema completo","sin_mentor":"Sin mentor"}
    fig, axes = plt.subplots(1, 2, figsize=(12,4))
    fig.suptitle("P2 — Efecto del mecanismo de mentor (ablación)",
                 fontsize=13, fontweight="bold")
    x = np.arange(len(dataset_names)); w = 0.35
    for ax, (metric, ylabel) in zip(axes, [("acc_final","Accuracy final"),("expq_final","Exp Quality final")]):
        for ci, cond in enumerate(conditions):
            vals, errs = [], []
            for dn in dataset_names:
                r = ablation_results.get((cond, dn))
                if r and not r["df"].empty:
                    last = r["df"]["iteration"].max()
                    col = "accuracy" if metric=="acc_final" else "exp_quality"
                    v = r["df"][r["df"]["iteration"]==last][col]
                    vals.append(v.mean()); errs.append(v.std())
                else: vals.append(np.nan); errs.append(0)
            bars = ax.bar(x + (ci-0.5)*w, vals, w, yerr=errs,
                          label=labels[cond], color=colors[cond], alpha=0.85, capsize=4)
            if cond == "sin_mentor":
                for xi, (v, bar) in enumerate(zip(vals, bars)):
                    comp = ablation_results.get(("completo", dataset_names[xi]))
                    if comp and not comp["df"].empty:
                        last = comp["df"]["iteration"].max()
                        col = "accuracy" if metric=="acc_final" else "exp_quality"
                        delta = v - comp["df"][comp["df"]["iteration"]==last][col].mean()
                        if not np.isnan(delta):
                            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                                    f"{delta:+.2f}", ha="center", va="bottom", fontsize=7, color="red")
        ax.set_xticks(x); ax.set_xticklabels(dataset_names)
        ax.set_ylabel(ylabel); ax.set_title(ylabel); ax.set_ylim(0,1.15)
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout(); _save(fig, "fig2_p2_mentor")

def fig3_shap_lime_agreement(all_hard, dataset_names):
    fig, axes = plt.subplots(1, 3, figsize=(16,4), sharey=True)
    fig.suptitle("P3 — Coordinación SHAP y LIME", fontsize=13, fontweight="bold")
    for ax, dn in zip(axes, dataset_names):
        frames = [r["df"] for (gn,d),r in all_hard.items() if d==dn and not r["df"].empty]
        if not frames: ax.set_title(dn); continue
        combined = pd.concat(frames)
        for col, color, label, ls in [
            ("shap_lime_agree","steelblue","SHAP=LIME","-"),
            ("exp_agreement","coral","Agreement","--"),
        ]:
            if col in combined.columns:
                g = combined.groupby("iteration")[col].agg(["mean","std"])
                ax.plot(g.index, g["mean"], color=color, linewidth=1.5,
                        marker="o", markersize=3, linestyle=ls, label=label)
                ax.fill_between(g.index, g["mean"]-g["std"].fillna(0),
                                g["mean"]+g["std"].fillna(0), color=color, alpha=0.12)
        ax.set_title(dn, fontweight="bold"); ax.set_xlabel("Iteración")
        ax.set_ylim(-0.05,1.05); ax.grid(True, alpha=0.3)
        if ax is axes[0]: ax.set_ylabel("Fracción de acuerdo"); ax.legend(fontsize=8)
    plt.tight_layout(); _save(fig, "fig3_p3_agreement")

def fig4_hard_vs_easy(all_hard, all_easy, dataset_names):
    group_names = list(GROUPS.keys())
    da = np.full((len(group_names), len(dataset_names)), np.nan)
    de = np.full_like(da, np.nan)
    for i,gn in enumerate(group_names):
        for j,dn in enumerate(dataset_names):
            rh, re = all_hard.get((gn,dn)), all_easy.get((gn,dn))
            if not rh or not re or rh["df"].empty or re["df"].empty: continue
            def lm(r,c): l=r["df"]["iteration"].max(); return r["df"][r["df"]["iteration"]==l][c].mean()
            da[i,j] = lm(re,"accuracy") - lm(rh,"accuracy")
            de[i,j] = lm(re,"exp_quality") - lm(rh,"exp_quality")
    glabels = [f"{gn}: "+"+".join(MODEL_LABELS[m][:3] for m in GROUPS[gn]) for gn in group_names]
    fig, axes = plt.subplots(1, 2, figsize=(14,4))
    fig.suptitle("P4 — Δ (fácil − frontera)", fontsize=13, fontweight="bold")
    for ax, data, title in zip(axes,[da,de],["Δ Accuracy","Δ Exp Quality"]):
        sns.heatmap(data, annot=True, fmt=".2f", cmap="RdYlGn", center=0,
                    vmin=-0.3, vmax=0.3, xticklabels=dataset_names,
                    yticklabels=glabels, ax=ax, linewidths=0.5, cbar_kws={"shrink":0.8})
        ax.set_title(title, fontweight="bold"); ax.tick_params(axis="y", labelsize=8)
    plt.tight_layout(); _save(fig, "fig4_p4_heatmap")

def fig5_ablation_cooldown_vote(ablation_results, dataset_names):
    conditions = ["completo","sin_mentor","sin_cooldown","voto_simple"]
    clabels = {"completo":"Completo","sin_mentor":"Sin mentor",
               "sin_cooldown":"Sin cooldown","voto_simple":"Voto simple"}
    colors = ["steelblue","salmon","gold","mediumpurple"]
    fig, axes = plt.subplots(1,2, figsize=(14,4))
    fig.suptitle("P5 — Ablación completa", fontsize=13, fontweight="bold")
    x = np.arange(len(dataset_names)); w = 0.18
    for ax, (metric, ylabel, ylim) in zip(axes,[
        ("stopped_at","Iteraciones hasta parada\n(-1=no convergió)",(-2,25)),
        ("acc_final","Accuracy final",(0,1.15)),
    ]):
        for ci,(cond,col) in enumerate(zip(conditions,colors)):
            vals = []
            for dn in dataset_names:
                r = ablation_results.get((cond,dn))
                if r and not r["df"].empty:
                    v = r["stopped_at"] if metric=="stopped_at" else \
                        r["df"][r["df"]["iteration"]==r["df"]["iteration"].max()]["accuracy"].mean()
                else: v = np.nan
                vals.append(v)
            ax.bar(x+(ci-len(conditions)/2+0.5)*w, vals, w, label=clabels[cond], color=col, alpha=0.85)
        ax.set_xticks(x); ax.set_xticklabels(dataset_names)
        ax.set_ylabel(ylabel); ax.set_title(ylabel.split("\n")[0])
        ax.set_ylim(*ylim); ax.legend(fontsize=7); ax.grid(True,alpha=0.3,axis="y")
    plt.tight_layout(); _save(fig, "fig5_p5_ablation")

def fig6_fidelity_by_model(all_hard):
    fid = {mn: {"shap":[],"lime":[]} for mn in MODEL_LABELS}
    for (gn,dn),r in all_hard.items():
        df = r["df"]
        if df.empty: continue
        for mn in GROUPS[gn]:
            sub = df[df["agent"]==f"{mn}_{gn}_{dn}"]
            if sub.empty: continue
            fid[mn]["shap"] += sub["shap_fidelity"].dropna().tolist()
            fid[mn]["lime"] += sub["lime_fidelity"].dropna().tolist()
    td = {}
    for mn,tp in MODEL_TYPE.items():
        if tp not in td: td[tp] = {"shap":[],"lime":[]}
        td[tp]["shap"] += fid[mn]["shap"]; td[tp]["lime"] += fid[mn]["lime"]
    types = list(td.keys())
    fig, axes = plt.subplots(1,2, figsize=(12,4), sharey=True)
    fig.suptitle("P6 — Fidelity por tipo de modelo", fontsize=13, fontweight="bold")
    for ax, exp, color in zip(axes, ("shap","lime"), ("steelblue","coral")):
        data = [td[tp][exp] for tp in types]
        bp = ax.boxplot(data, labels=types, patch_artist=True, notch=False, showfliers=False)
        for patch in bp["boxes"]: patch.set_facecolor(color); patch.set_alpha(0.65)
        ax.set_title(f"Fidelity {exp.upper()}", fontweight="bold")
        ax.set_ylim(0,1.05); ax.tick_params(axis="x",rotation=15,labelsize=8)
        ax.grid(True,alpha=0.3,axis="y")
        if ax is axes[0]: ax.set_ylabel("Fidelity")
    plt.tight_layout(); _save(fig, "fig6_p6_fidelity")

def tabla1_summary(all_hard, dataset_names):
    rows = []
    for (gn,dn),r in all_hard.items():
        df = r["df"]
        if df.empty: continue
        last = df["iteration"].max(); dfl = df[df["iteration"]==last]
        cv = list(r["convergence"].values())
        mia = np.mean([v["iter_acc"] for v in cv if v["iter_acc"]>=0]) if any(v["iter_acc"]>=0 for v in cv) else -1
        mie = np.mean([v["iter_exp"] for v in cv if v["iter_exp"]>=0]) if any(v["iter_exp"]>=0 for v in cv) else -1
        rows.append({"Grupo":gn,"Dataset":dn,
                     "Acc final":round(dfl["accuracy"].mean(),3),
                     "ExpQ final":round(dfl["exp_quality"].mean(),3),
                     "Fid SHAP":round(df["shap_fidelity"].mean(),3),
                     "Fid LIME":round(df["lime_fidelity"].mean(),3),
                     "Agreement":round(df["exp_agreement"].mean(),3),
                     "Stop iter":r["stopped_at"],
                     "W.diff":r["weighted_diff"],
                     "Mentor":r["n_mentor"],"Cooldown":r["n_cooldown"],
                     "Conv acc":round(mia,1) if mia>=0 else "-",
                     "Conv expq":round(mie,1) if mie>=0 else "-"})
    return pd.DataFrame(rows).sort_values(["Grupo","Dataset"]).reset_index(drop=True)

# ── Plot apéndice ─────────────────────────────────────────────────────────────

AGENT_COLORS = ["steelblue","coral","seagreen"]

def plot_run_appendix(result, group_name, model_names, dataset_label,
                      instance_label, instance_idx, true_label,
                      condition="completo"):
    df = result["df"]
    if df.empty: return
    agent_ids    = result["classifier_ids"]
    mentor_iters = result.get("mentor_iters", set())
    n_agents     = len(agent_ids)
    glabel = f"{group_name} [{condition}]: " + " + ".join(MODEL_LABELS[m] for m in model_names)
    decision_map = {"keep":0,"soft_adjust":1,"adjust":2,"force_adjust":3}

    fig, axes = plt.subplots(n_agents, 4, figsize=(18, 3.8*n_agents),
                              constrained_layout=True)
    if n_agents == 1: axes = axes[np.newaxis, :]
    fig.suptitle(f"{glabel}  ×  {dataset_label}  —  "
                 f"{instance_label} (idx={instance_idx}, clase={true_label})",
                 fontsize=10, fontweight="bold")

    for ri, (aid, mn) in enumerate(zip(agent_ids, model_names)):
        sub = df[df["agent"]==aid].reset_index(drop=True)
        if sub.empty: continue
        it  = sub["iteration"]
        col = AGENT_COLORS[ri % len(AGENT_COLORS)]

        # Col 0: Accuracy + ExpQ
        ax = axes[ri,0]
        ax.plot(it, sub["accuracy"],    color=col, marker="o", markersize=3, lw=1.4, label="Accuracy")
        ax.plot(it, sub["exp_quality"], color=col, marker="s", markersize=3, lw=1.4, ls="--", alpha=0.7, label="ExpQ")
        ax.axhline(CONV_ACC_THRESHOLD, color="steelblue", ls=":", alpha=0.4)
        ax.axhline(CONV_EXP_THRESHOLD, color="coral",     ls=":", alpha=0.4)
        if result["stopped_at"] >= 0:
            ax.axvline(result["stopped_at"], color="red", ls=":", lw=1.2, label=f"Stop@{result['stopped_at']}")
        ax.set_title(MODEL_LABELS[mn], fontsize=9, fontweight="bold")
        ax.set_ylabel("Score"); ax.set_ylim(0,1.05); ax.set_xlabel("Iteración")
        ax.legend(fontsize=6); ax.grid(True, alpha=0.25)

        # Col 1: Fidelity SHAP/LIME
        ax = axes[ri,1]
        ax.plot(it, sub["shap_fidelity"], color="steelblue", marker="o", markersize=3, lw=1.4, label="SHAP")
        ax.plot(it, sub["lime_fidelity"], color="coral",     marker="s", markersize=3, lw=1.4, ls="--", label="LIME")
        ax.plot(it, sub["exp_fidelity"],  color="gray",      marker="D", markersize=2, lw=1,   ls="-.", alpha=0.5, label="Media")
        if "shap_lime_agree" in sub.columns:
            ax2 = ax.twinx()
            ax2.fill_between(it, sub["shap_lime_agree"].astype(float), alpha=0.08, color="gold")
            ax2.set_ylim(0,4); ax2.set_yticks([])
        ax.set_title("Fidelity SHAP/LIME", fontsize=9); ax.set_ylabel("Fidelity")
        ax.set_ylim(0,1.05); ax.set_xlabel("Iteración"); ax.legend(fontsize=6); ax.grid(True, alpha=0.25)

        # Col 2: Consensus/Stability/Agreement
        ax = axes[ri,2]
        ax.plot(it, sub["exp_consensus"], color="purple", marker="o", markersize=3, lw=1.4, label="Consensus")
        ax.plot(it, sub["exp_stability"], color="orange", marker="s", markersize=3, lw=1.4, ls="--", label="Stability")
        ax.plot(it, sub["exp_agreement"], color="teal",   marker="^", markersize=3, lw=1.4, ls=":",  label="Agreement")
        ax.axhline(0.65, color="gray", ls=":", alpha=0.4)
        ax.set_title("Consensus/Stability/Agreement", fontsize=9); ax.set_ylabel("Score")
        ax.set_ylim(0,1.05); ax.set_xlabel("Iteración"); ax.legend(fontsize=6); ax.grid(True, alpha=0.25)

        # Col 3: Decisiones + mentor + predicción
        ax = axes[ri,3]
        ax.step(it, sub["decision"].map(decision_map).fillna(0),
                where="mid", color="darkblue", lw=1.4)
        for mit in mentor_iters:
            ax.axvline(mit, color="gold", alpha=0.6, lw=1.5, ls="--")
        ax.set_yticks([0,1,2,3]); ax.set_yticklabels(["keep","soft","adj","force"], fontsize=7)
        ax.set_ylabel("Estrategia"); ax.set_xlabel("Iteración")
        axp = ax.twinx()
        axp.step(it, sub["prediction"], where="mid", color="indigo", lw=1.2, ls="-.", alpha=0.7)
        axp.axhline(true_label, color="green", ls=":", lw=1, alpha=0.7)
        axp.set_ylabel("Pred/Real", fontsize=7); axp.tick_params(axis="y", labelsize=7)
        ax.set_title("Decisiones/Predicción", fontsize=9); ax.grid(True, alpha=0.25)

    tag  = f"{group_name}_{condition}_{result['dataset_name']}_{instance_label}"
    path = f"appendix/{tag}.png"
    fig.savefig(path, bbox_inches="tight", dpi=120)
    plt.close(fig)
    print(f"    → {path}")

# ── Orquestador ───────────────────────────────────────────────────────────────

async def run_all_experiments():
    print("\n" + "="*65)
    print(f"  TFG — MAX_ITER={MAX_ITER}  |  LIME reducido por dataset")
    print("="*65)

    print("\n[Cargando datasets...]")
    datasets = load_datasets(); dataset_names = list(datasets.keys())
    scaled = {}; instances = {}

    for dn, info in datasets.items():
        sc = StandardScaler(); X_sc = sc.fit_transform(info["X_raw"])
        hi, ei = select_instances(X_sc, info["y"])
        scaled[dn] = X_sc
        instances[dn] = {"hard_idx":hi,"easy_idx":ei,
                          "hard_inst":X_sc[hi:hi+1],"easy_inst":X_sc[ei:ei+1],
                          "hard_label":int(info["y"][hi]),"easy_label":int(info["y"][ei])}
        print(f"  {dn:12s} | frontera={hi} clase={info['y'][hi]}"
              f" | fácil={ei} clase={info['y'][ei]}")

    # FASE 1
    print("\n" + "─"*65)
    print("  FASE 1 — Base (5 grupos × 3 datasets × 2 instancias)")
    print("─"*65)
    all_hard, all_easy = {}, {}

    for gn, model_names in GROUPS.items():
        for dn, info in datasets.items():
            inst = instances[dn]
            for ikey, iarr, ilabel in [("hard",inst["hard_inst"],"frontera"),
                                        ("easy",inst["easy_inst"],"fácil")]:
                idx   = inst[f"{ikey}_idx"]
                label = inst[f"{ikey}_label"]
                print(f"  ▶ {gn} × {dn} × {ilabel}")
                try:
                    r = await run_single(gn, model_names, dn,
                                         scaled[dn], info["y"],
                                         info["feature_names"], iarr, ilabel)
                    (all_hard if ikey=="hard" else all_easy)[(gn,dn)] = r
                    plot_run_appendix(r, gn, model_names, info["label"],
                                      ilabel, idx, label)
                except Exception as ex:
                    print(f"    [ERROR] {ex}")

    # FASE 2
    print("\n" + "─"*65)
    print(f"  FASE 2 — Ablación ({ABLATION_GROUP} × 3 datasets × 3 cond)")
    print("─"*65)
    ablation = {}
    for dn in dataset_names:
        if (ABLATION_GROUP, dn) in all_hard:
            ablation[("completo", dn)] = all_hard[(ABLATION_GROUP, dn)]

    abl_models = GROUPS[ABLATION_GROUP]
    for cond, flags in {
        "sin_mentor":   {"mentor_enabled":False,"cooldown_enabled":True, "weighted_vote":True},
        "sin_cooldown": {"mentor_enabled":True, "cooldown_enabled":False,"weighted_vote":True},
        "voto_simple":  {"mentor_enabled":True, "cooldown_enabled":True, "weighted_vote":False},
    }.items():
        for dn, info in datasets.items():
            inst = instances[dn]
            print(f"  ▶ {cond} × {dn}")
            try:
                r = await run_single(ABLATION_GROUP, abl_models, dn,
                                      scaled[dn], info["y"],
                                      info["feature_names"],
                                      inst["hard_inst"], "frontera", **flags)
                ablation[(cond, dn)] = r
                plot_run_appendix(r, ABLATION_GROUP, abl_models, info["label"],
                                  "frontera", inst["hard_idx"],
                                  inst["hard_label"], condition=cond)
            except Exception as ex:
                print(f"    [ERROR] {ex}")

    # Figuras
    print("\n" + "─"*65)
    print("  FIGURAS")
    print("─"*65)
    for fn, args in [
        (fig1_explainability_evolution, (all_hard, dataset_names)),
        (fig2_mentor_ablation,          (ablation, dataset_names)),
        (fig3_shap_lime_agreement,      (all_hard, dataset_names)),
        (fig4_hard_vs_easy,             (all_hard, all_easy, dataset_names)),
        (fig5_ablation_cooldown_vote,   (ablation, dataset_names)),
        (fig6_fidelity_by_model,        (all_hard,)),
    ]:
        print(f"  {fn.__name__}")
        fn(*args)

    print("\n  Tabla 1\n")
    tabla = tabla1_summary(all_hard, dataset_names)
    print(tabla.to_string(index=False))
    return all_hard, all_easy, ablation, tabla

# ── Guardado ──────────────────────────────────────────────────────────────────

def save_results(all_hard, all_easy, ablation, tabla, output_dir="results"):
    os.makedirs(output_dir, exist_ok=True)
    tabla.to_csv(f"{output_dir}/tabla_resumen.csv", index=False)
    print("  ✓ tabla_resumen.csv")

    def collect(results, label):
        fs = []
        for (gn,dn),r in results.items():
            if r["df"].empty: continue
            df = r["df"].copy(); df["grupo"]=gn; df["dataset"]=dn; df["condicion"]=label
            fs.append(df)
        return pd.concat(fs).reset_index(drop=True) if fs else pd.DataFrame()

    for df, fname in [(collect(all_hard,"frontera"),"base_hard.csv"),
                       (collect(all_easy,"facil"),   "base_easy.csv")]:
        df.to_csv(f"{output_dir}/{fname}", index=False)
        print(f"  ✓ {fname}  ({len(df)} filas)")

    afs = []
    for (cond,dn),r in ablation.items():
        if r["df"].empty: continue
        df=r["df"].copy(); df["condicion"]=cond; df["dataset"]=dn; afs.append(df)
    if afs:
        abl = pd.concat(afs).reset_index(drop=True)
        abl.to_csv(f"{output_dir}/ablation.csv", index=False)
        print(f"  ✓ ablation.csv  ({len(abl)} filas)")

    conv_rows = []
    for (gn,dn),r in all_hard.items():
        for aid,cv in r["convergence"].items():
            conv_rows.append({"grupo":gn,"dataset":dn,"agent":aid,
                               "iter_acc":cv["iter_acc"],"iter_exp":cv["iter_exp"],
                               "order":cv["order"]})
    if conv_rows:
        pd.DataFrame(conv_rows).to_csv(f"{output_dir}/convergencia.csv", index=False)
        print("  ✓ convergencia.csv")

    mc = []
    for (gn,dn),r in {**all_hard, **{(f"abl_{c}",d):v for (c,d),v in ablation.items()}}.items():
        mc.append({"grupo":gn,"dataset":dn,"instancia":r.get("instance_label","frontera"),
                    "stopped_at":r["stopped_at"],"n_mentor":r["n_mentor"],
                    "n_dissent":r["n_dissent"],"n_cooldown":r["n_cooldown"],
                    "weighted_diff":r["weighted_diff"]})
    if mc:
        pd.DataFrame(mc).to_csv(f"{output_dir}/mentor_cooldown.csv", index=False)
        print("  ✓ mentor_cooldown.csv")

    print(f"\n  Resultados en '{output_dir}/'")

# ── Punto de entrada ──────────────────────────────────────────────────────────

import json

def serialize_results(all_hard, all_easy, ablation, tabla):
    output = {}
    
    # Tabla resumen principal
    output["tabla_resumen"] = tabla.to_dict(orient="records")
    
    # Por cada run: métricas finales, convergencia, mentor, cooldown
    output["runs_hard"] = {}
    for (gn, dn), r in all_hard.items():
        key = f"{gn}__{dn}"
        df = r["df"]
        if df.empty:
            continue
        last = df["iteration"].max()
        dfl = df[df["iteration"] == last]
        output["runs_hard"][key] = {
            "stopped_at": r["stopped_at"],
            "n_mentor": r["n_mentor"],
            "n_dissent": r["n_dissent"],
            "n_cooldown": r["n_cooldown"],
            "weighted_diff": r["weighted_diff"],
            "convergence": r["convergence"],
            "final_metrics": dfl.groupby("agent")[
                ["accuracy","exp_quality","exp_fidelity",
                 "exp_consensus","exp_stability","exp_agreement",
                 "shap_fidelity","lime_fidelity","shap_lime_agree"]
            ].mean().round(4).to_dict(orient="index"),
            "evolution_by_iter": df.groupby("iteration")[
                ["accuracy","exp_quality","exp_fidelity",
                 "exp_consensus","exp_stability","exp_agreement"]
            ].mean().round(4).to_dict(orient="index"),
        }
    
    output["runs_easy"] = {}
    for (gn, dn), r in all_easy.items():
        key = f"{gn}__{dn}"
        df = r["df"]
        if df.empty:
            continue
        last = df["iteration"].max()
        dfl = df[df["iteration"] == last]
        output["runs_easy"][key] = {
            "stopped_at": r["stopped_at"],
            "final_metrics": dfl.groupby("agent")[
                ["accuracy","exp_quality","exp_fidelity","exp_consensus"]
            ].mean().round(4).to_dict(orient="index"),
        }

    output["ablation"] = {}
    for (cond, dn), r in ablation.items():
        key = f"{cond}__{dn}"
        df = r["df"]
        if df.empty:
            continue
        last = df["iteration"].max()
        dfl = df[df["iteration"] == last]
        output["ablation"][key] = {
            "stopped_at": r["stopped_at"],
            "n_mentor": r["n_mentor"],
            "n_cooldown": r["n_cooldown"],
            "final_metrics": dfl.groupby("agent")[
                ["accuracy","exp_quality","exp_fidelity","exp_consensus"]
            ].mean().round(4).to_dict(orient="index"),
        }

    with open("results/resultados_cap4.json", "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2, default=str)
    print("\n  ✓ results/resultados_cap4.json")


all_hard, all_easy, ablation, tabla = await run_all_experiments()
serialize_results(all_hard, all_easy, ablation, tabla)
save_results(all_hard, all_easy, ablation, tabla)


  TFG — MAX_ITER=20  |  LIME reducido por dataset

[Cargando datasets...]
  wine         | frontera=83 clase=1 | fácil=1 clase=0
  digits       | frontera=421 clase=5 | fácil=0 clase=0
  covertype    | frontera=1288 clase=4 | fácil=68 clase=1

─────────────────────────────────────────────────────────────────
  FASE 1 — Base (5 grupos × 3 datasets × 2 instancias)
─────────────────────────────────────────────────────────────────
  ▶ G1 × wine × frontera
[rf_G1_wine] Setup iniciado
[rf_G1_wine] 11:39:54 | Setup iniciado (train + eval inicial)
[rf_G1_wine] 11:39:54 | Entrenando modelo (iter=0)
[rf_G1_wine] 11:39:54 | Evaluación completada | acc=0.972
[rf_G1_wine] 11:39:54 | Setup completado
[gb_G1_wine] Setup iniciado
[gb_G1_wine] 11:39:54 | Setup iniciado (train + eval inicial)
[gb_G1_wine] 11:39:54 | Entrenando modelo (iter=0)
[gb_G1_wine] 11:39:54 | Evaluación completada | acc=0.972
[gb_G1_wine] 11:39:54 | Setup completado
[torch_G1_wine] Setup iniciado
[torch_G1_wine] 11:39:54 | Setup

AttributeError: 'DataFrame' object has no attribute 'ro'